# Setup

In [ ]:
import os

In [ ]:
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

## Extract Raw Data

In [ ]:
# 2. Configuration
RAW_DATA_PATH = "/content/drive/MyDrive/Engage_Raw"

# 3. Change directory to where the files are
try:
    os.chdir(RAW_DATA_PATH)
    print(f"📂 Changed directory to: {os.getcwd()}")
except FileNotFoundError:
    print(f"❌ Error: Could not find directory: {RAW_DATA_PATH}")
    print("Please verify the folder name in your Google Drive.")

# 4. Verify the compressed files are actually here
print("\nChecking for compressed files...")
files_in_dir = os.listdir('.')
if 'reddit-data-01-part-01.tar.gz' in files_in_dir:
    print("✅ Found part 1")
else:
    print("❌ Missing part 1")

if 'reddit-data-01-part-02.tar.gz' in files_in_dir:
    print("✅ Found part 2")
else:
    print("❌ Missing part 2")

# 5. Combine the two parts (Only run if files exist)
print("\nCombining file parts... (This takes 1-2 mins)")
!cat reddit-data-01-part-01.tar.gz reddit-data-01-part-02.tar.gz > reddit.tar.gz

# 6. Extract the JSON files
print("Extracting files... (This takes 5-10 mins)")
!tar -xzf reddit.tar.gz

# 7. Verify extraction
json_files = sorted([f for f in os.listdir('.') if f.endswith('.json') and f.startswith('data')])
print(f"\n✅ Success! Found {len(json_files)} JSON data files.")
if len(json_files) > 0:
    print(f"First file: {json_files[0]}")
    print(f"Last file: {json_files[-1]}")

In [ ]:
!pip install scikit-learn numpy tqdm

# V3.5

In [ ]:
%%writefile process_engage_corpus_v3_5_chi2.py
#!/usr/bin/env python3
import json
import os
import re
from collections import defaultdict, Counter
import numpy as np
from tqdm import tqdm
import argparse
import random
import pickle
from math import log
import gc  # Added for memory management

# Chi-Squared for Feature Selection
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2

# Configuration - MUST match v3
RANDOM_SEED = 42
NUM_SUBREDDITS = 5000
START_2019 = 1546300800
CONTEXT_END = 1561939199
VW_TARGET_START = 1561939200
VW_TARGET_END = 1569887999
EVAL_START = 1569888000
END_2019 = 1577836799

def simple_tokenize(text):
    if not text: return []
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text: return []
    return text.split()

def load_all_data_files(data_dir):
    json_files = sorted([f for f in os.listdir(data_dir) if f.startswith('data') and f.endswith('.json')])
    print(f"Found {len(json_files)} data files")
    for json_file in tqdm(json_files, desc="Loading data files"):
        with open(os.path.join(data_dir, json_file), 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        yield json.loads(line)
                    except json.JSONDecodeError: continue

def filter_user_by_periods(user):
    periods = {'context': [], 'vw_target': [], 'ncf_train': [], 'eval': []}
    # (Simplified logic to save space, functionality identical to original)
    for item_type in ['posts', 'comments']:
        for item in user.get(item_type, []):
            ts = item.get('created_utc', 0)
            if START_2019 <= ts <= CONTEXT_END:
                periods['context'].append(item); periods['ncf_train'].append(item)
            elif VW_TARGET_START <= ts <= VW_TARGET_END:
                periods['vw_target'].append(item); periods['ncf_train'].append(item)
            elif EVAL_START <= ts <= END_2019:
                periods['eval'].append(item)
    return {k: {'posts': [x for x in v if 'title' in x], 'comments': [x for x in v if 'body' in x]}
            for k,v in periods.items()}

def pass1_collect_subreddit_stats(data_dir, output_dir):
    cache_file = os.path.join(output_dir, 'cache_subreddit_stats.pkl')
    if os.path.exists(cache_file):
        print("PASS 1: Loading cached subreddit statistics")
        with open(cache_file, 'rb') as f: return pickle.load(f)

    print("PASS 1: Collecting subreddit statistics")
    subreddit_users = defaultdict(set)
    for user in load_all_data_files(data_dir):
        user_num = user['user_number']
        for item in user.get('posts', []) + user.get('comments', []):
            subreddit_users[item.get('subreddit')].add(user_num)

    subreddit_counts = sorted([(s, len(u)) for s, u in subreddit_users.items()], key=lambda x: x[1], reverse=True)
    top_5000 = [s for s, c in subreddit_counts[:NUM_SUBREDDITS]]
    result = (set(top_5000), top_5000)
    with open(cache_file, 'wb') as f: pickle.dump(result, f)
    return result

def pass2_select_users(data_dir, top_subs, output_dir):
    cache_file = os.path.join(output_dir, 'cache_selected_users.pkl')
    if os.path.exists(cache_file):
        print("PASS 2: Loading cached selected users")
        with open(cache_file, 'rb') as f: return pickle.load(f)

    print("PASS 2: Selecting 6% of users")
    eligible = []
    for user in load_all_data_files(data_dir):
        # Quick check for any interaction in top subs
        all_items = user.get('posts', []) + user.get('comments', [])
        if any(i.get('subreddit') in top_subs for i in all_items):
            eligible.append(user['user_number'])

    random.seed(RANDOM_SEED)
    selected = set(random.sample(eligible, int(len(eligible) * 0.06)))
    with open(cache_file, 'wb') as f: pickle.dump(selected, f)
    return selected

def pass3_compute_word_stats(data_dir, top_subs, selected_users, output_dir):
    cache_file = os.path.join(output_dir, 'cache_word_stats.pkl')
    if os.path.exists(cache_file):
        print("PASS 3: Loading cached word statistics")
        with open(cache_file, 'rb') as f: return pickle.load(f)

    print("PASS 3: Computing word statistics")
    # Using smaller int types for counts could save memory, but sticking to standard for now
    word_doc_freq = Counter()
    total_docs = 0
    user_word_freq_train = defaultdict(Counter)
    user_word_freq_dev = defaultdict(Counter)
    user_word_freq_test = defaultdict(Counter)
    subreddit_word_freq_train = defaultdict(Counter)
    subreddit_word_freq_dev = defaultdict(Counter)
    subreddit_word_freq_test = defaultdict(Counter)
    user_num_to_id = {}
    user_id_counter = 0
    skipped = 0

    for user in tqdm(load_all_data_files(data_dir), desc="Computing word stats"):
        if user['user_number'] not in selected_users: continue

        # Simplified period filtering for brevity in this memory-safe version
        # (Logic matches original script exactly)
        eval_items = [i for i in user.get('posts', []) + user.get('comments', [])
                     if EVAL_START <= i.get('created_utc', 0) <= END_2019]
        if len(eval_items) < 2:
            skipped += 1; continue

        uid = user_id_counter; user_num_to_id[user['user_number']] = uid; user_id_counter += 1

        # Train words
        train_items = [i for i in user.get('posts', []) + user.get('comments', [])
                      if START_2019 <= i.get('created_utc', 0) <= VW_TARGET_END]
        train_words = set()
        for item in train_items:
            tokens = simple_tokenize(item.get('title', '') + ' ' + item.get('selftext', '') + ' ' + item.get('body', ''))
            for t in tokens:
                user_word_freq_train[uid][t] += 1
                if item.get('subreddit') in top_subs: subreddit_word_freq_train[item['subreddit']][t] += 1
            train_words.update(tokens)

        for w in train_words: word_doc_freq[w] += 1
        total_docs += 1

        # Eval words
        eval_items.sort(key=lambda x: x['created_utc'])
        for idx, target_dict, target_sub_dict in [(0, user_word_freq_dev, subreddit_word_freq_dev),
                                                  (1, user_word_freq_test, subreddit_word_freq_test)]:
            item = eval_items[idx]
            tokens = simple_tokenize(item.get('title', '') + ' ' + item.get('selftext', '') + ' ' + item.get('body', ''))
            for t in tokens:
                target_dict[uid][t] += 1
                if item.get('subreddit') in top_subs: target_sub_dict[item['subreddit']][t] += 1

    result = {
        'word_doc_freq': word_doc_freq, 'total_docs': total_docs,
        'user_word_freq_train': dict(user_word_freq_train),
        'user_word_freq_dev': dict(user_word_freq_dev),
        'user_word_freq_test': dict(user_word_freq_test),
        'subreddit_word_freq_train': dict(subreddit_word_freq_train),
        'subreddit_word_freq_dev': dict(subreddit_word_freq_dev),
        'subreddit_word_freq_test': dict(subreddit_word_freq_test),
        'user_num_to_id': user_num_to_id, 'skipped_users': skipped
    }
    with open(cache_file, 'wb') as f: pickle.dump(result, f)
    return result

# --- NEW BATCHED CHI2 FUNCTION ---
def compute_global_chi2_scores(subreddit_word_freq_train):
    print("Computing global Chi-Square scores (BATCHED MODE)...")

    # 1. Vectorize (Sparse) - Low Memory
    vectorizer = DictVectorizer(sparse=True)
    subreddits = sorted(subreddit_word_freq_train.keys())
    dict_list = [subreddit_word_freq_train[sub] for sub in subreddits]

    print("  Vectorizing data...")
    X = vectorizer.fit_transform(dict_list)
    feature_names = vectorizer.get_feature_names_out()
    y = list(range(len(dict_list))) # Each subreddit is its own class

    # 2. Compute Chi2 in Batches
    n_features = X.shape[1]
    batch_size = 5000 # Process 5000 words at a time to stay under RAM limit
    all_chi2_stats = []

    print(f"  Matrix shape: {X.shape}. Processing in batches of {batch_size}...")

    for i in range(0, n_features, batch_size):
        end = min(i + batch_size, n_features)
        X_batch = X[:, i:end]

        # sklearn's chi2 works efficiently on sparse input if 'y' matches rows
        chi2_stats, _ = chi2(X_batch, y)
        all_chi2_stats.extend(chi2_stats)

        if i % 100000 == 0 and i > 0:
            print(f"  Processed {i}/{n_features} words...")
            gc.collect() # Force cleanup

    word_scores = dict(zip(feature_names, all_chi2_stats))
    max_score = max(word_scores.values()) if word_scores else 1.0
    for w in word_scores: word_scores[w] /= max_score

    print(f"  Done! Computed scores for {len(word_scores)} words.")
    return word_scores

def filter_words_by_score(word_freq, global_scores, top_k=50):
    scored = [(w, global_scores.get(w, 0.0)) for w in word_freq]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [w for w, s in scored[:top_k]]

def pass4_generate_outputs(data_dir, top_subs, sub_list, selected_users, word_stats, output_dir):
    print("PASS 4: Generating outputs")

    # Unpack stats (Free memory of unused items)
    sub_word_freq = word_stats['subreddit_word_freq_train']
    user_num_to_id = word_stats['user_num_to_id']

    # Run Batched Chi2
    global_scores = compute_global_chi2_scores(sub_word_freq)

    # Create Dirs
    for d in ['text_context', 'text_context_filtered', 'ncf_data']:
        os.makedirs(os.path.join(output_dir, d), exist_ok=True)

    # Helper to save json
    def save_json(data, name, subdir):
        with open(os.path.join(output_dir, subdir, name), 'w') as f: json.dump(data, f)

    # Process Users (Train/Dev/Test)
    for phase in ['train', 'dev', 'test']:
        print(f"  Processing User {phase} text...")
        raw_dict = {}; filt_dict = {}
        source_freq = word_stats[f'user_word_freq_{phase}']

        for uid, freq in source_freq.items():
            raw_dict[uid] = ' '.join([w for w in freq for _ in range(freq[w])])
            filt_dict[uid] = ' '.join(filter_words_by_score(freq, global_scores))

        save_json(raw_dict, f'user_text_{phase}.json', 'text_context')
        save_json(filt_dict, f'user_text_{phase}.json', 'text_context_filtered')

        # Free memory
        del source_freq; del raw_dict; del filt_dict; gc.collect()

    # Process Subreddits
    for phase in ['train', 'dev', 'test']:
        print(f"  Processing Subreddit {phase} text...")
        raw_dict = {}; filt_dict = {}
        source_freq = word_stats[f'subreddit_word_freq_{phase}']

        for sub, freq in source_freq.items():
            raw_dict[sub] = ' '.join([w for w in freq for _ in range(freq[w])])
            filt_dict[sub] = ' '.join(filter_words_by_score(freq, global_scores))

        save_json(raw_dict, f'subreddit_text_{phase}.json', 'text_context')
        save_json(filt_dict, f'subreddit_text_{phase}.json', 'text_context_filtered')
        del source_freq; gc.collect()

    # Interactions (NCF Data)
    print("  Generating NCF interactions...")
    # (Re-loading to avoid passing huge dicts around)
    sub2idx = {s: i for i, s in enumerate(sub_list)}

    # Using specific logic for v3.5 (from original script)
    train_f = open(os.path.join(output_dir, 'ncf_data', 'train.tsv'), 'w')
    dev_f = open(os.path.join(output_dir, 'ncf_data', 'dev.tsv'), 'w')
    test_f = open(os.path.join(output_dir, 'ncf_data', 'test.tsv'), 'w')

    for user in load_all_data_files(data_dir):
        if user['user_number'] not in user_num_to_id: continue
        uid = user_num_to_id[user['user_number']]

        # Original v3.5 interaction logic
        # 1. Train
        seen = set()
        for item in user.get('posts', []) + user.get('comments', []):
            if START_2019 <= item.get('created_utc', 0) <= VW_TARGET_END:
                s = item.get('subreddit')
                if s in sub2idx and s not in seen:
                    train_f.write(f"{uid}\t{sub2idx[s]}\n"); seen.add(s)

        # 2. Eval
        evals = [i for i in user.get('posts', []) + user.get('comments', [])
                 if EVAL_START <= i.get('created_utc', 0) <= END_2019]
        evals.sort(key=lambda x: x['created_utc'])
        if len(evals) >= 2:
            s1 = evals[0].get('subreddit')
            s2 = evals[1].get('subreddit')
            if s1 in sub2idx: dev_f.write(f"{uid}\t{sub2idx[s1]}\n")
            if s2 in sub2idx: test_f.write(f"{uid}\t{sub2idx[s2]}\n")

    train_f.close(); dev_f.close(); test_f.close()

    # Mappings
    with open(os.path.join(output_dir, 'user_mapping.json'), 'w') as f:
        json.dump({'user_num_to_id': {str(k):v for k,v in user_num_to_id.items()}, 'num_users': len(user_num_to_id)}, f)
    with open(os.path.join(output_dir, 'subreddit_mapping.json'), 'w') as f:
        json.dump({'subreddits': sub_list, 'subreddit2idx': sub2idx, 'num_subreddits': len(sub_list)}, f)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data_dir', required=True)
    parser.add_argument('--output_dir', default='engage_corpus_processed_v3_5_chi2')
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    top_subs, sub_list = pass1_collect_subreddit_stats(args.data_dir, args.output_dir)
    selected_users = pass2_select_users(args.data_dir, top_subs, args.output_dir)
    word_stats = pass3_compute_word_stats(args.data_dir, top_subs, selected_users, args.output_dir)
    pass4_generate_outputs(args.data_dir, top_subs, sub_list, selected_users, word_stats, args.output_dir)

if __name__ == "__main__":
    main()

# V4

In [ ]:
%%writefile process_engage_corpus_v4_chi2.py
#!/usr/bin/env python3
import json
import os
import re
from collections import defaultdict, Counter
import numpy as np
from tqdm import tqdm
import argparse
import random
import pickle
import gc
import shutil

from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2

# Config
RANDOM_SEED = 42
NUM_SUBREDDITS = 2000
START_2019 = 1546300800
TRAIN_INPUT_END = 1561939199
TRAIN_TARGET_START = 1561939200
TRAIN_TARGET_END = 1569887999
DEV_START_OCT = 1569888000
DEV_END_OCT = 1572566399
TEST_START_NOV = 1572566400
TEST_END_NOV = 1575158399
DEV_START_DEC = 1575158400
END_2019 = 1577836799

def simple_tokenize(text):
    if not text: return []
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text: return []
    return text.split()

def load_all_data_files(data_dir):
    json_files = sorted([f for f in os.listdir(data_dir) if f.startswith('data') and f.endswith('.json')])
    for json_file in tqdm(json_files, desc="Loading data files"):
        with open(os.path.join(data_dir, json_file), 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        yield json.loads(line)
                    except json.JSONDecodeError: continue

def filter_user_periods_v4(user):
    periods = {'train_input': [], 'train_target': [], 'dev': [], 'test': []}
    for item in user.get('posts', []) + user.get('comments', []):
        ts = item.get('created_utc', 0)
        if START_2019 <= ts <= TRAIN_INPUT_END: periods['train_input'].append(item)
        elif TRAIN_TARGET_START <= ts <= TRAIN_TARGET_END: periods['train_target'].append(item)
        elif (DEV_START_OCT <= ts <= DEV_END_OCT) or (DEV_START_DEC <= ts <= END_2019): periods['dev'].append(item)
        elif TEST_START_NOV <= ts <= TEST_END_NOV: periods['test'].append(item)
    return periods

def load_cache_robust(filepath, description):
    if os.path.exists(filepath):
        print(f"{description}: Found cache at {filepath}")
        try:
            with open(filepath, 'rb') as f:
                return pickle.load(f)
        except Exception:
            print(f"⚠ WARNING: Cache corrupted. Deleting {filepath}...")
            try: os.remove(filepath)
            except: pass
    return None

def pass1_collect_subreddit_stats(data_dir, output_dir):
    cache_file = os.path.join(output_dir, 'cache_subreddit_stats.pkl')
    cached = load_cache_robust(cache_file, "PASS 1")
    if cached: return cached

    print("PASS 1: Collecting stats")
    subreddit_users = defaultdict(set)
    for user in load_all_data_files(data_dir):
        uid = user['user_number']
        for item in user.get('posts', []) + user.get('comments', []):
            subreddit_users[item.get('subreddit')].add(uid)
    counts = sorted([(s, len(u)) for s,u in subreddit_users.items()], key=lambda x:x[1], reverse=True)
    top_k = [s for s,c in counts[:NUM_SUBREDDITS]]
    res = (set(top_k), top_k)
    with open(cache_file, 'wb') as f: pickle.dump(res, f)
    return res

def pass2_select_users(data_dir, top_subs, output_dir):
    cache_file = os.path.join(output_dir, 'cache_selected_users.pkl')
    cached = load_cache_robust(cache_file, "PASS 2")
    if cached: return cached

    print("PASS 2: Selecting users")
    selected = set()
    for user in load_all_data_files(data_dir):
        p = filter_user_periods_v4(user)
        if not (len(p['train_input']) > 0 and len(p['train_target']) > 0): continue
        if not (50 <= (len(p['train_input']) + len(p['train_target'])) <= 5000): continue
        if any(i.get('subreddit') in top_subs for i in p['train_input'] + p['train_target']):
            selected.add(user['user_number'])
    with open(cache_file, 'wb') as f: pickle.dump(selected, f)
    return selected

def pass3_compute_word_stats_streaming(data_dir, top_subs, selected_users, output_dir,
                                       debug_max_users=None):
    """
    STREAMING VERSION: Writes user stats directly to .jsonl files to avoid RAM OOM.
    Only keeps subreddit stats in memory (much smaller).
    """
    cache_meta = os.path.join(output_dir, 'cache_stats_meta.pkl')
    cache_sub_stats = os.path.join(output_dir, 'cache_stats_subreddits.pkl')

    # Check if we are done
    if os.path.exists(cache_meta) and os.path.exists(cache_sub_stats):
        print("PASS 3: Found streaming caches. Skipping computation.")
        return True

    print("PASS 3: Computing word stats (Streaming Mode)")

    # Files to stream user data to
    f_train = open(os.path.join(output_dir, 'temp_user_train.jsonl'), 'w')
    f_dev = open(os.path.join(output_dir, 'temp_user_dev.jsonl'), 'w')
    f_test = open(os.path.join(output_dir, 'temp_user_test.jsonl'), 'w')

    # Keep Global stats in RAM (manageable)
    word_doc_freq = Counter()
    sub_word_freq_train = defaultdict(Counter)
    sub_word_freq_dev = defaultdict(Counter)
    sub_word_freq_test = defaultdict(Counter)

    user_num_to_id = {}; uid_ctr = 0

    try:
        for user in tqdm(load_all_data_files(data_dir)):
            if user['user_number'] not in selected_users: continue

            # DEBUG LIMIT: stop after processing debug_max_users selected users
            if debug_max_users is not None and uid_ctr >= debug_max_users:
                break

            uid = uid_ctr; user_num_to_id[user['user_number']] = uid; uid_ctr += 1
            p = filter_user_periods_v4(user)

            # --- PROCESS TRAIN ---
            train_words = set()
            u_freq = Counter()
            for item in p['train_input'] + p['train_target']:
                tokens = simple_tokenize(item.get('title','')+ ' '+ item.get('selftext','')+ ' '+ item.get('body',''))
                for t in tokens:
                    u_freq[t] += 1
                    if item.get('subreddit') in top_subs: sub_word_freq_train[item['subreddit']][t] += 1
                train_words.update(tokens)

            # Write User Stats Immediately & Flush
            if u_freq:
                f_train.write(json.dumps({'uid': uid, 'freq': dict(u_freq)}) + '\n')

            for w in train_words: word_doc_freq[w] += 1

            # --- PROCESS DEV ---
            u_freq = Counter()
            for item in p['dev']:
                tokens = simple_tokenize(item.get('title','')+ ' '+ item.get('selftext','')+ ' '+ item.get('body',''))
                for t in tokens:
                    u_freq[t] += 1
                    if item.get('subreddit') in top_subs: sub_word_freq_dev[item['subreddit']][t] += 1
            if u_freq:
                f_dev.write(json.dumps({'uid': uid, 'freq': dict(u_freq)}) + '\n')

            # --- PROCESS TEST ---
            u_freq = Counter()
            for item in p['test']:
                tokens = simple_tokenize(item.get('title','')+ ' '+ item.get('selftext','')+ ' '+ item.get('body',''))
                for t in tokens:
                    u_freq[t] += 1
                    if item.get('subreddit') in top_subs: sub_word_freq_test[item['subreddit']][t] += 1
            if u_freq:
                f_test.write(json.dumps({'uid': uid, 'freq': dict(u_freq)}) + '\n')

        # Close files
        f_train.close(); f_dev.close(); f_test.close()

        # Save Meta and Subreddit Stats
        print("Saving Meta Stats...")
        with open(cache_meta, 'wb') as f:
            pickle.dump({'word_doc_freq': word_doc_freq, 'user_num_to_id': user_num_to_id}, f)

        print("Saving Subreddit Stats...")
        with open(cache_sub_stats, 'wb') as f:
            pickle.dump({
                'sub_train': sub_word_freq_train,
                'sub_dev': sub_word_freq_dev,
                'sub_test': sub_word_freq_test
            }, f)

        return True

    except Exception as e:
        print(f"CRASH DETECTED: {e}")
        f_train.close(); f_dev.close(); f_test.close()
        raise e

def compute_global_chi2_scores(subreddit_word_freq_train):
    print("Computing global Chi-Square scores (BATCHED MODE)...")
    vectorizer = DictVectorizer(sparse=True)
    subreddits = sorted(subreddit_word_freq_train.keys())
    dict_list = [subreddit_word_freq_train[sub] for sub in subreddits]

    print("  Vectorizing data...")
    X = vectorizer.fit_transform(dict_list)
    feature_names = vectorizer.get_feature_names_out()
    y = list(range(len(dict_list)))

    n_features = X.shape[1]
    batch_size = 5000
    all_chi2_stats = []

    print(f"  Matrix shape: {X.shape}. Processing in batches...")
    for i in range(0, n_features, batch_size):
        end = min(i + batch_size, n_features)
        X_batch = X[:, i:end]
        chi2_stats, _ = chi2(X_batch, y)
        all_chi2_stats.extend(chi2_stats)
        if i % 100000 == 0: gc.collect()

    word_scores = dict(zip(feature_names, all_chi2_stats))
    max_score = max(word_scores.values()) if word_scores else 1.0
    for w in word_scores: word_scores[w] /= max_score
    return word_scores

def filter_words_by_score(word_freq, global_scores, top_k=50):
    scored = [(w, global_scores.get(w, 0.0)) for w in word_freq]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [w for w, s in scored[:top_k]]

def save_json(data, name, subdir, output_dir):
    with open(os.path.join(output_dir, subdir, name), 'w') as f: json.dump(data, f)

def process_streamed_users(jsonl_path, global_scores, output_name, output_dir):
    """Reads streaming JSONL, filters, and writes final JSON output."""
    if not os.path.exists(jsonl_path):
        print(f"  Note: No data for {output_name}")
        save_json({}, output_name, 'text_context', output_dir)
        save_json({}, output_name, 'text_context_filtered', output_dir)
        return

    print(f"  Processing {output_name} from stream...")
    raw_dict = {}
    filt_dict = {}

    with open(jsonl_path, 'r') as f:
        for line in f:
            entry = json.loads(line)
            uid = str(entry['uid']) # JSON keys must be strings
            freq = entry['freq']

            # Reconstruct raw string (repeating words)
            raw_dict[uid] = ' '.join([w for w in freq for _ in range(freq[w])])
            # Filter
            filt_dict[uid] = ' '.join(filter_words_by_score(freq, global_scores))

    save_json(raw_dict, output_name, 'text_context', output_dir)
    save_json(filt_dict, output_name, 'text_context_filtered', output_dir)

    # Delete temp file to free space
    os.remove(jsonl_path)

def pass4_generate_outputs(data_dir, top_subs, sub_list, selected_users, output_dir,
                           debug_max_users=None):
    print("PASS 4: Generating outputs (Incremental Load)")

    for d in ['text_context', 'text_context_filtered', 'ncf_data']:
        os.makedirs(os.path.join(output_dir, d), exist_ok=True)

    # 1. Load Subreddit Stats & Compute Chi2
    print("  Loading Subreddit stats...")
    with open(os.path.join(output_dir, 'cache_stats_subreddits.pkl'), 'rb') as f:
        sub_stats = pickle.load(f)

    global_scores = compute_global_chi2_scores(sub_stats['sub_train'])

    # 2. Process Users (Reading from temp streams)
    process_streamed_users(os.path.join(output_dir, 'temp_user_train.jsonl'), global_scores, 'user_text_train.json', output_dir)
    process_streamed_users(os.path.join(output_dir, 'temp_user_dev.jsonl'), global_scores, 'user_text_dev.json', output_dir)
    process_streamed_users(os.path.join(output_dir, 'temp_user_test.jsonl'), global_scores, 'user_text_test.json', output_dir)

    # 3. Process Subreddits (From memory/cache)
    print("  Processing Subreddit text...")
    for phase in ['train', 'dev', 'test']:
        raw = {}; filt = {}
        for sub, freq in sub_stats[f'sub_{phase}'].items():
            raw[sub] = ' '.join([w for w in freq for _ in range(freq[w])])
            filt[sub] = ' '.join(filter_words_by_score(freq, global_scores))
        save_json(raw, f'subreddit_text_{phase}.json', 'text_context', output_dir)
        save_json(filt, f'subreddit_text_{phase}.json', 'text_context_filtered', output_dir)

    del sub_stats; gc.collect()

    # 4. Generate Interactions (Needs User Mapping)
    print("  Loading User Mapping...")
    with open(os.path.join(output_dir, 'cache_stats_meta.pkl'), 'rb') as f:
        meta = pickle.load(f)
    user_num_to_id = meta['user_num_to_id']
    del meta; gc.collect()

    print("  Generating interactions...")
    sub2idx = {s: i for i, s in enumerate(sub_list)}

    train_f = open(os.path.join(output_dir, 'ncf_data', 'train.tsv'), 'w')
    dev_f = open(os.path.join(output_dir, 'ncf_data', 'dev.tsv'), 'w')
    test_f = open(os.path.join(output_dir, 'ncf_data', 'test.tsv'), 'w')

    processed_users = 0
    for user in load_all_data_files(data_dir):
        if user['user_number'] not in user_num_to_id: continue

        # DEBUG LIMIT: stop after writing interactions for debug_max_users users
        if debug_max_users is not None and processed_users >= debug_max_users:
            break
        processed_users += 1

        uid = user_num_to_id[user['user_number']]
        p = filter_user_periods_v4(user)

        def write_interactions(items, file_handle):
            seen = set()
            for i in items:
                s = i.get('subreddit')
                if s in sub2idx and s not in seen:
                    file_handle.write(f"{uid}\t{sub2idx[s]}\t1\n"); seen.add(s)

        write_interactions(p['train_target'], train_f)
        write_interactions(p['dev'], dev_f)
        write_interactions(p['test'], test_f)

    train_f.close(); dev_f.close(); test_f.close()

    # Save Final Mappings
    with open(os.path.join(output_dir, 'user_mapping.json'), 'w') as f:
        json.dump({'user_num_to_id': {str(k):v for k,v in user_num_to_id.items()}, 'num_users': len(user_num_to_id)}, f)
    with open(os.path.join(output_dir, 'subreddit_mapping.json'), 'w') as f:
        json.dump({'subreddits': sub_list, 'subreddit2idx': sub2idx, 'num_subreddits': len(sub_list)}, f)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data_dir', required=True)
    parser.add_argument('--output_dir', default='engage_corpus_processed_v4_chi2')
    parser.add_argument('--debug_max_users', type=int, default=None,
                        help="If set, limit number of users processed (for fast debugging).")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    top_subs, sub_list = pass1_collect_subreddit_stats(args.data_dir, args.output_dir)
    selected_users = pass2_select_users(args.data_dir, top_subs, args.output_dir)

    # Pass 3: Computes stats
    pass3_compute_word_stats_streaming(
        args.data_dir, top_subs, selected_users, args.output_dir,
        debug_max_users=args.debug_max_users
    )

    # Pass 4: Generate outputs
    pass4_generate_outputs(
        args.data_dir, top_subs, sub_list, selected_users, args.output_dir,
        debug_max_users=args.debug_max_users
    )

if __name__ == "__main__":
    main()


# Run Processing

In [ ]:
# 1. Configuration - THE CORRECT PATH YOU FOUND
RAW_DATA_PATH = "/content/drive/MyDrive/Engage_Raw/converted_data"
OUTPUT_PARENT_DIR = "/content/drive/MyDrive/CIS 5300"

# 2. Define Output Paths (mimicking original structure)
v4_output = os.path.join(OUTPUT_PARENT_DIR, "engage_corpus_processed_v4_chi2")
v35_output = os.path.join(OUTPUT_PARENT_DIR, "engage_corpus_processed_v3_5_chi2")

## V3.5

In [ ]:
# 4. Run v3.5 Processing (Smaller Dataset - For Prototyping)
print(f"🚀 Starting v3.5 Processing (Chi-Square)...")
print(f"   Input: {RAW_DATA_PATH}")
print(f"   Output: {v35_output}")
!python process_engage_corpus_v3_5_chi2.py \
    --data_dir "$RAW_DATA_PATH" \
    --output_dir "$v35_output"

## V4

## Retry

In [ ]:
RAW_DATA_PATH = "/content/drive/MyDrive/Engage_Raw/converted_data"
OUTPUT_BASE = "/content/drive/MyDrive/CIS 5300"

print("🚀 Restarting v3.5 with Batched Chi2...")
v35_out = os.path.join(OUTPUT_BASE, "engage_corpus_processed_v3_5_chi2")
!python process_engage_corpus_v3_5_chi2.py --data_dir "$RAW_DATA_PATH" --output_dir "$v35_out"

In [ ]:
# 1. Define the corrupted file path
v4_output_dir = "/content/drive/MyDrive/CIS 5300/engage_corpus_processed_v4_chi2"
bad_cache = os.path.join(v4_output_dir, "cache_word_stats.pkl")

# 2. Delete it
if os.path.exists(bad_cache):
    print(f"🗑️ Found corrupted cache: {bad_cache}")
    try:
        os.remove(bad_cache)
        print("✅ Deleted successfully. The script will now regenerate it.")
    except OSError as e:
        print(f"❌ Error deleting file: {e}")
else:
    print("ℹ️ Cache file not found (it might have been deleted already).")

# 3. Restart v4 Processing
RAW_DATA_PATH = "/content/drive/MyDrive/Engage_Raw/converted_data"

print(f"\n🚀 RESTARTING v4 PROCESSING...")
!python process_engage_corpus_v4_chi2.py \
    --data_dir "$RAW_DATA_PATH" \
    --output_dir "$v4_output_dir"

In [ ]:
!python process_engage_corpus_v4_chi2.py \
  --data_dir "$RAW_DATA_PATH" \
  --output_dir "$v4_output_dir"
